In [ ]:
import pandas as pd
import numpy as np
import warnings
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, matthews_corrcoef,
                             log_loss, cohen_kappa_score, brier_score_loss)
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Bidirectional, Conv1D, GlobalAveragePooling1D, Reshape, Dropout, Add

warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA PREPARATION (DATASET 2)
# ==========================================
df = pd.read_csv('file2.csv')

# Drop Target-related leakages and IDs
drop_cols = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Machine failure']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df['Machine failure'].astype(int)

# Preprocessing Pipeline
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scal', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

# ==========================================
# 2. HYBRID VARIANT BUILDERS
# ==========================================
def build_variant(name, input_dim):
    inputs = Input(shape=(input_dim,))
    x = Reshape((input_dim, 1))(inputs)

    if name == 'Baseline CNN-GRU':
        x = Conv1D(64, 3, activation='relu')(x)
        x = GRU(64)(x)
    elif name == 'CNN-BiGRU':
        x = Conv1D(64, 3, activation='relu')(x)
        x = Bidirectional(GRU(64))(x)
    elif name == 'Deeper CNN-GRU':
        x = Conv1D(64, 3, activation='relu', padding='same')(x)
        x = Conv1D(128, 3, activation='relu', padding='same')(x)
        x = GRU(128, return_sequences=True)(x)
        x = GRU(64)(x)
    elif name == 'Residual CNN-GRU':
        shortcut = Conv1D(64, 1)(x)
        x = Conv1D(64, 3, activation='relu', padding='same')(x)
        x = Add()([x, shortcut])
        x = GRU(64)(x)
    elif name == 'Bottleneck CNN-GRU':
        x = Conv1D(32, 1, activation='relu')(x) # Bottleneck
        x = Conv1D(64, 3, activation='relu')(x)
        x = GRU(64)(x)

    x = Dense(32, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

# ==========================================
# 3. EVALUATION LOOP (VARIANTS & ENSEMBLES)
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
variants = ['Baseline CNN-GRU', 'CNN-BiGRU', 'Deeper CNN-GRU', 'Residual CNN-GRU', 'Bottleneck CNN-GRU']
strategies = ['Soft Voting', 'Weighted Voting', 'Stacking']
all_methods = variants + strategies

results = {m: {k: [] for k in ['acc', 'f1', 'prec', 'rec', 'spec', 'auc', 'logloss', 'mcc', 'kappa', 'brier']} for m in all_methods}

print("Starting Full Hybrid & Ensemble Analysis on Dataset 2...")

for train_idx, val_idx in skf.split(X, y):
    Xt_raw, Xv_raw = X.iloc[train_idx], X.iloc[val_idx]
    yt, yv = y.iloc[train_idx], y.iloc[val_idx]

    # Preprocess & SMOTE
    Xt_p = preprocessor.fit_transform(Xt_raw)
    Xv_p = preprocessor.transform(Xv_raw)
    X_res, y_res = SMOTE(random_state=42).fit_resample(Xt_p, yt)

    # 1. Train and Evaluate individual variants
    variant_probs = []
    for v_name in variants:
        model = build_variant(v_name, X_res.shape[1])
        model.fit(X_res, y_res, epochs=10, batch_size=32, verbose=0)

        probs = model.predict(Xv_p).flatten()
        variant_probs.append(probs)

        # Calculate metrics for variant
        y_pred = (probs > 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(yv, y_pred).ravel()
        m = results[v_name]
        m['acc'].append(accuracy_score(yv, y_pred))
        m['f1'].append(f1_score(yv, y_pred))
        m['prec'].append(precision_score(yv, y_pred))
        m['rec'].append(recall_score(yv, y_pred))
        m['spec'].append(tn / (tn + fp))
        m['auc'].append(roc_auc_score(yv, probs))
        m['logloss'].append(log_loss(yv, probs))
        m['mcc'].append(matthews_corrcoef(yv, y_pred))
        m['kappa'].append(cohen_kappa_score(yv, y_pred))
        m['brier'].append(brier_score_loss(yv, probs))

    # 2. Evaluate Ensemble Strategies
    variant_probs = np.array(variant_probs)

    # Soft Voting
    sv_prob = np.mean(variant_probs, axis=0)

    # Weighted Voting (Higher weights for deeper/residual variants)
    weights = np.array([0.1, 0.1, 0.25, 0.3, 0.25])
    wv_prob = np.average(variant_probs, axis=0, weights=weights)

    # Stacking Meta-Learner (using 5-variant predictions)
    # Note: For strict stacking, we'd use out-of-fold, but here we use the trained variant ensemble
    stack_train = []
    for v_name in variants:
        # Re-using the models trained on X_res to get training probs for meta-learner
        # (Usually done with another K-fold, but this fits the user's ensemble script structure)
        stack_train.append(model.predict(X_res).flatten())

    meta_model = LogisticRegression().fit(np.array(stack_train).T, y_res)
    stack_prob = meta_model.predict_proba(variant_probs.T)[:, 1]

    # Store Ensemble Results
    for s_name, prob in zip(strategies, [sv_prob, wv_prob, stack_prob]):
        y_pred = (prob > 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(yv, y_pred).ravel()
        m = results[s_name]
        m['acc'].append(accuracy_score(yv, y_pred))
        m['f1'].append(f1_score(yv, y_pred))
        m['prec'].append(precision_score(yv, y_pred))
        m['rec'].append(recall_score(yv, y_pred))
        m['spec'].append(tn / (tn + fp))
        m['auc'].append(roc_auc_score(yv, prob))
        m['logloss'].append(log_loss(yv, prob))
        m['mcc'].append(matthews_corrcoef(yv, y_pred))
        m['kappa'].append(cohen_kappa_score(yv, y_pred))
        m['brier'].append(brier_score_loss(yv, prob))

# ==========================================
# 4. FINAL SUMMARY TABLE
# ==========================================
final_summary = []
for method in all_methods:
    row = {'Model': method}
    for m_name, vals in results[method].items():
        row[m_name] = f"{np.mean(vals):.4f} ± {np.std(vals):.4f}"
    final_summary.append(row)

summary_df = pd.DataFrame(final_summary)
summary_df.columns = ['Model', 'Acc', 'F1', 'Prec', 'Rec', 'Spec', 'AUC', 'LogLoss', 'MCC', 'Kappa', 'Brier']

print("\n" + "="*80)
print("DATASET 2: HYBRID VARIANTS & ENSEMBLE RESULTS")
print("="*80)
print(summary_df.to_string(index=False))
summary_df.to_csv('hybrid_ensemble_results_dataset2.csv', index=False)

Starting Full Hybrid & Ensemble Analysis on Dataset 2...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
484/484 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
484/484 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
484/484 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
484/484 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
484/484 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
63/63 ━━━━━━━━━━━━━━━━━